# Solve MaxCut with QAOA

Encode a triangle as a QUBO and optimize a QAOA circuit. Recover a candidate cut, compare its discrete energy with the expected objective, and draw the optimized circuit.

Install QASP 0.1.3 as shown in the [tutorial setup guide](../README.md), then run the cells in order.

In [ ]:
%matplotlib inline
from importlib.metadata import version
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

print("QASP", version("qasp"))
np.random.seed(17)
torch.manual_seed(17)

## Configure the experiment

Each edge contributes when its endpoints are assigned different bits. The standard parametrization has one cost angle and one mixer angle per layer. QAOA takes the problem Hamiltonian as its input, so its configuration uses only init.

In [ ]:
from qasp import Normal
from qasp import QUBO, QAOA, draw_qaoa

problem_config = {
    "n_variables": 3,
    "edges": [(0, 1), (1, 2), (0, 2)],
}
config = {
    "init": Normal(0, 0.2),
    "n_layers": 1, "parametrization": "standard",
    "device": "default.qubit", "backend": "torch",
    "diff_method": "backprop", "shots": None, "seed": 7,
}
training_config = {
    "optimizer": "Adam",
    "optimizer_options": {"maxiter": 18, "lr": 0.05},
    "include_probabilities": True,
}

## Construct and inspect the initial model

Run this cell again after editing the configuration to sample fresh parameters.

In [ ]:
problem = QUBO.maxcut(**problem_config)
model = QAOA(problem, **config)
print("Qubits:", model.n_qubits, "parameters:", model.n_parameters)
start = model.initial_parameters()
print("Starting angles:", start)
print("Starting expectation:", model.expectation(start))

## Train

Construct the starting vector once and pass that exact vector to solve. Adam updates the continuous angles; the final distribution yields candidate bitstrings.

In [ ]:
result = model.solve(initial_parameters=start, **training_config)
print("Optimization history:", result.history)
assert np.isfinite(result.parameters).all()

## Read the results

The reported energy belongs to a particular bitstring. The expectation objective averages energies over the circuit distribution. This small graph permits a direct check of the candidate against every bitstring.

In [ ]:
print("Candidate:", result.bitstring)
print("QUBO energy:", result.energy)
print("Expectation objective:", result.objective)
print("Candidate probability:", result.solution_probability)
print("Distribution:", result.probabilities)
print("Optimizer:", result.success, result.message)
assert np.isclose(problem.energy(result.bitstring), result.energy)

In [ ]:
from itertools import product
best = min(problem.energy(bits) for bits in product((0, 1), repeat=3))
print("Best discrete energy:", best)
print("Candidate energy gap:", result.energy - best)

## Inspect the circuit

Use the trained weights and the same representative input for both renderers. Matplotlib is the default for the module drawing helpers.

In [ ]:
fig, ax = draw_qaoa(model, parameters=result, measurement="probabilities")
display(fig)
plt.close(fig)

### Preview the paper figure

Request standalone TikZ for a vector PDF. Symbolic labels describe the circuit architecture. Set `symbolic_parameters=False` to inspect the gate angles at this input.

`draw_qaoa` displays the circuit inline. Use `return_tex=False` for a figure-only cell. To also get the LaTeX, omit that argument and assign the drawing call to `tex`. The preview uses temporary files.

In [ ]:
draw_qaoa(
    model, parameters=result, measurement="probabilities", output="tikz", return_tex=False,
    tikz_options={"standalone": True, "symbolic_parameters": True, "row_sep": "0.7cm"},
)

## Try it

Increase the depth to two layers, reconstruct the initial vector, and compare solution probability at the same optimization budget.